# Built-in Tools: File Editing

The SDK ships built-in `Read`, `Write`, and `Edit` tools out of the box — no custom tool code needed. `ClaudeSDKClient` with default options already has these available.


In [10]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()  # this notebook lives in episodes/, project root is one level up
SCRATCH_DIR = PROJECT_ROOT / "04_notes"
NOTES_PATH = SCRATCH_DIR / "notes.txt"


def write_scratch_notes() -> None:
    # Just plain Python — create a small text file we'll ask Claude to edit below.
    SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
    NOTES_PATH.write_text("Line 1: placeholder note.\nLine 2: another placeholder note.\n")


write_scratch_notes()
print("--- before ---")
print(NOTES_PATH.read_text())

--- before ---
Line 1: placeholder note.
Line 2: another placeholder note.



In [11]:
from claude_agent_sdk import (
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
    AssistantMessage,  # message type that holds Claude's actual reply
    ToolUseBlock,  # message piece that shows "Claude is calling a tool now"
)


async def edit_notes_file() -> None:
    # allowed_tools: list of tool names Claude is allowed to use WITHOUT asking
    # for permission first. Read/Edit/Write are built into the SDK already —
    # we're just pre-approving them so this notebook can run without a manual
    # "yes, allow this" prompt popping up.
    options = ClaudeAgentOptions(model="haiku", allowed_tools=["Read", "Edit", "Write"])

    async with ClaudeSDKClient(options=options) as client:
        # We describe the task in plain English. Claude decides on its own
        # to call the Read tool (to see the file) and then the Edit tool
        # (to add the new line) — we never call those tools ourselves.
        await client.query(f"Read the file {NOTES_PATH} and append one new line to it: " "'Line 3: added by Claude.'")
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock):
                        print(f"[tool call] {block.name}({block.input})")
            if isinstance(message, ResultMessage):
                print(message.result)


await edit_notes_file()
print("--- after ---")
print(NOTES_PATH.read_text())

[tool call] Read({'file_path': '/Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/episodes/04_notes/notes.txt'})
[tool call] Edit({'replace_all': False, 'file_path': '/Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/episodes/04_notes/notes.txt', 'old_string': 'Line 1: placeholder note.\nLine 2: another placeholder note.\n', 'new_string': 'Line 1: placeholder note.\nLine 2: another placeholder note.\nLine 3: added by Claude.\n'})
Done! I've successfully appended the line 'Line 3: added by Claude.' to the file `/Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/episodes/04_notes/notes.txt`. The file now contains the original two lines plus the new line you requested.
--- after ---
Line 1: placeholder note.
Line 2: another placeholder note.
Line 3: added by Claude.



## Cleanup

Per this project's CLAUDE.md, scratch files under `02_notes` get deleted as soon as the task using them is done — not left around between episodes.


In [12]:
# Plain Python cleanup — delete the scratch file and directory we created for this demo.
NOTES_PATH.unlink(missing_ok=True)
print(f"Deleted {NOTES_PATH}")

if SCRATCH_DIR.exists():
    remaining = [p.name for p in SCRATCH_DIR.iterdir()]
    print("02_notes now contains:", remaining)
    if not remaining:
        SCRATCH_DIR.rmdir()
        print(f"Removed empty directory {SCRATCH_DIR}")
else:
    print("02_notes does not exist — nothing to clean up.")

Deleted /Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/episodes/04_notes/notes.txt
02_notes now contains: []
Removed empty directory /Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/episodes/04_notes


## Summary

- Built-in file tools (`Read`, `Write`, `Edit`) work with zero setup on `ClaudeSDKClient`.
- Always clean up scratch files the agent touched once the episode/task is done.
